In [3]:
from function import *
import cv2
import numpy as np
import mediapipe as mp
from keras.models import model_from_json

# Load model
with open("model.json", "r") as json_file:
    model_json = json_file.read()

model = model_from_json(model_json)
model.load_weights("model.h5")

# Define actions (⚠️ UPDATE THIS)
actions = np.array(['A','B','C','D','E','F','G','H','I','J',
                    'K','L','M','N','O','P','Q','R','S','T'])

# Colors
colors = [(245, 117, 16)] * len(actions)

# Detection variables
sequence = []
sentence = []
accuracy = []
predictions = []
threshold = 0.8

# Start webcam
cap = cv2.VideoCapture(0)

mp_hands = mp.solutions.hands

with mp_hands.Hands(
    model_complexity=0,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5) as hands:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Flip for mirror view (optional)
        frame = cv2.flip(frame, 1)

        # Define ROI
        cropframe = frame[40:400, 0:300]

        # Draw detection box
        cv2.rectangle(frame, (0, 40), (300, 400), (255, 0, 0), 2)

        # Mediapipe detection
        image, results = mediapipe_detection(cropframe, hands)

        # Extract keypoints
        keypoints = extract_keypoints(results)
        sequence.append(keypoints)
        sequence = sequence[-30:]

        try:
            if len(sequence) == 30:
                res = model.predict(np.expand_dims(sequence, axis=0))[0]
                predicted_class = np.argmax(res)
                predictions.append(predicted_class)

                # Smooth predictions
                if len(predictions) >= 10 and len(set(predictions[-10:])) == 1:
                    if res[predicted_class] > threshold:
                        if len(sentence) == 0 or actions[predicted_class] != sentence[-1]:
                            sentence.append(actions[predicted_class])
                            accuracy.append(f"{res[predicted_class]*100:.2f}%")

                # Keep only latest
                sentence = sentence[-1:]
                accuracy = accuracy[-1:]

        except Exception as e:
            print("Error:", e)

        # ================= UI ================= #

        # 🔵 Top blue bar
        cv2.rectangle(frame, (0, 0), (frame.shape[1], 50), (255, 0, 0), -1)

        current_text = ""
        if len(sentence) > 0:
            current_text = f"Current: {sentence[-1]} ({accuracy[-1]})"

        cv2.putText(frame, current_text, (10, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

        # ⚫ Bottom black bar
        cv2.rectangle(frame, (0, frame.shape[0] - 50),
                      (frame.shape[1], frame.shape[0]), (0, 0, 0), -1)

        recognized_text = ""
        if len(sentence) > 0:
            recognized_text = f"Recognized: {sentence[-1]}"

        cv2.putText(frame, recognized_text, (10, frame.shape[0] - 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

        # Show frame
        cv2.imshow('OpenCV Feed', frame)

        # Exit
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

Error: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (30,) + inhomogeneous part.
Error: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (30,) + inhomogeneous part.
Error: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (30,) + inhomogeneous part.
Error: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (30,) + inhomogeneous part.
Error: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (30,) + inhomogeneous part.
Error: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (30,) + inhomogeneous part

In [5]:
from function import *
import cv2
import numpy as np
import mediapipe as mp
from keras.models import model_from_json

# ================= LOAD MODEL ================= #
with open("model.json", "r") as json_file:
    model_json = json_file.read()

model = model_from_json(model_json)
model.load_weights("model.h5")

# ✅ MUST MATCH TRAINING CLASSES
actions = np.array([
    'A','B','C','D','E','F','G','H','I','J','K','L','M',
    'N','O','P','Q','R','S','T','U','V','W','X','Y','Z',
    '0','1','2','3','4','5','6','7','8','9',
    ' ','THANKYOU','HELLO','HI','YES','NO'
])

# ================= VARIABLES ================= #
sequence = []
sentence = []
accuracy = []
predictions = []
threshold = 0.8

# Webcam
cap = cv2.VideoCapture(0)

mp_hands = mp.solutions.hands

# ================= MAIN LOOP ================= #
with mp_hands.Hands(
    model_complexity=0,
    max_num_hands=2,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7) as hands:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)

        # ROI
        cropframe = frame[40:400, 0:300]

        # Blue box
        cv2.rectangle(frame, (0, 40), (300, 400), (255, 0, 0), 2)

        # Detection
        image, results = mediapipe_detection(cropframe, hands)

        # Keypoints (FIXED SIZE = 126)
        keypoints = extract_keypoints(results)
        sequence.append(keypoints)
        sequence = sequence[-30:]

        # ================= PREDICTION ================= #
        if len(sequence) == 30:
            try:
                sequence_np = np.array(sequence)
                res = model.predict(np.expand_dims(sequence_np, axis=0))[0]

                predicted_class = np.argmax(res)
                predictions.append(predicted_class)

                # Smooth predictions
                if len(predictions) >= 10 and len(set(predictions[-10:])) == 1:
                    if res[predicted_class] > threshold:
                        if len(sentence) == 0 or actions[predicted_class] != sentence[-1]:
                            sentence.append(actions[predicted_class])
                            accuracy.append(f"{res[predicted_class]*100:.2f}%")

                sentence = sentence[-1:]
                accuracy = accuracy[-1:]

            except Exception as e:
                print("Prediction Error:", e)

        # ================= UI ================= #

        # 🔵 Top bar
        cv2.rectangle(frame, (0, 0), (frame.shape[1], 50), (255, 0, 0), -1)

        if len(sentence) > 0:
            current_text = f"Current: {sentence[-1]} ({accuracy[-1]})"
        else:
            current_text = "Current: -"

        cv2.putText(frame, current_text, (10, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

        # ⚫ Bottom bar
        cv2.rectangle(frame,
                      (0, frame.shape[0] - 50),
                      (frame.shape[1], frame.shape[0]),
                      (0, 0, 0), -1)

        if len(sentence) > 0:
            recognized_text = f"Recognized: {sentence[-1]}"
        else:
            recognized_text = "Recognized: -"

        cv2.putText(frame, recognized_text,
                    (10, frame.shape[0] - 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

        # ❗ No hand warning
        if not results.multi_hand_landmarks:
            cv2.putText(frame, "No Hand Detected",
                        (320, 100),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                        (0, 0, 255), 2)

        # Show
        cv2.imshow('ASL Recognition', frame)

        # Exit
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

Prediction Error: Invalid dtype: object
Prediction Error: Invalid dtype: object
Prediction Error: Invalid dtype: object
Prediction Error: Invalid dtype: object
Prediction Error: Invalid dtype: object
Prediction Error: Invalid dtype: object
Prediction Error: Invalid dtype: object
Prediction Error: Invalid dtype: object
Prediction Error: Invalid dtype: object
Prediction Error: Invalid dtype: object
Prediction Error: Invalid dtype: object
Prediction Error: Invalid dtype: object
Prediction Error: Invalid dtype: object
Prediction Error: Invalid dtype: object
Prediction Error: Invalid dtype: object
Prediction Error: Invalid dtype: object
Prediction Error: Invalid dtype: object
Prediction Error: Invalid dtype: object
Prediction Error: Invalid dtype: object
Prediction Error: Invalid dtype: object
Prediction Error: Invalid dtype: object
Prediction Error: Invalid dtype: object
Prediction Error: Invalid dtype: object
Prediction Error: Invalid dtype: object
Prediction Error: Invalid dtype: object


In [17]:
from function import *
import cv2
import numpy as np
import mediapipe as mp
from keras.models import model_from_json

# ================= LOAD MODEL ================= #
with open("model.json", "r") as json_file:
    model_json = json_file.read()

model = model_from_json(model_json)
model.load_weights("model.h5")

# ⚠️ MUST match training classes
actions = np.array([
    'A','B','C','D','E','F','G','H','I','J',
    'K','L','M','N','O','P','Q','R','S','T','U','V','W','X','Y','Z'
])

# ================= VARIABLES ================= #
sequence = []
predictions = []
threshold = 0.8

word = ""
final_text = ""

# Webcam
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)  # 🔥 reduce lag

mp_hands = mp.solutions.hands

# ================= MAIN LOOP ================= #
with mp_hands.Hands(
    model_complexity=0,
    max_num_hands=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7) as hands:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)

        # ROI
        cropframe = frame[40:400, 0:300]

        # Blue box
        cv2.rectangle(frame, (0, 40), (300, 400), (255, 0, 0), 2)

        # Detection
        image, results = mediapipe_detection(cropframe, hands)

        # Keypoints
        keypoints = extract_keypoints(results)
        sequence.append(keypoints)
        sequence = sequence[-15:]   # ⚡ faster (was 30)

        # ================= PREDICTION ================= #
        if len(sequence) == 15:
            try:
                seq = np.array(sequence)
                res = model.predict(np.expand_dims(seq, axis=0), verbose=0)[0]

                pred = np.argmax(res)
                confidence = res[pred]
                predictions.append(pred)

                # 🔥 faster smoothing
                if len(predictions) >= 5 and len(set(predictions[-5:])) == 1:
                    if confidence > threshold:
                        char = actions[pred]

                        # avoid duplicates
                        if len(word) == 0 or char != word[-1]:
                            word += char

                # keep last few predictions
                predictions = predictions[-10:]

            except Exception as e:
                print("Prediction Error:", e)

        # ================= KEY CONTROLS ================= #
        key = cv2.waitKey(10) & 0xFF

        if key == ord('q'):
            break

        elif key == ord(' '):  # add word
            final_text += word + " "
            word = ""

        elif key == ord('c'):  # clear
            word = ""
            final_text = ""

        # ================= UI ================= #

        # 🔵 Top bar
        cv2.rectangle(frame, (0, 0), (frame.shape[1], 50), (255, 0, 0), -1)

        if len(word) > 0:
            current_text = f"Current: {word[-1]}"
        else:
            current_text = "Current: -"

        cv2.putText(frame, current_text, (10, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

        # ⚫ Bottom bar
        cv2.rectangle(frame,
                      (0, frame.shape[0] - 50),
                      (frame.shape[1], frame.shape[0]),
                      (0, 0, 0), -1)

        recognized_text = f"Recognized: {final_text + word}"

        cv2.putText(frame, recognized_text,
                    (10, frame.shape[0] - 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

        # ❗ No hand
        if not results.multi_hand_landmarks:
            cv2.putText(frame, "No Hand Detected",
                        (320, 100),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                        (0, 0, 255), 2)

        cv2.imshow("ASL Recognition", frame)

    cap.release()
    cv2.destroyAllWindows()

Prediction Error: Invalid dtype: object
Prediction Error: Invalid dtype: object
Prediction Error: Invalid dtype: object
Prediction Error: Invalid dtype: object
Prediction Error: Invalid dtype: object
Prediction Error: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (15,) + inhomogeneous part.
Prediction Error: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (15,) + inhomogeneous part.
Prediction Error: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (15,) + inhomogeneous part.
Prediction Error: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (15,) + inhomogeneous part.
Prediction Error: setting an array element with a sequence. The requested array has an i